### Vector stores and retrievers
This video tutorial will familiarize you with LangChain's vector store and retriever abstractions. These abstractions are designed to support retrieval of data-- from (vector) databases and other sources-- for integration with LLM workflows. They are important for applications that fetch data to be reasoned over as part of model inference, as in the case of retrieval-augmented generation.

We will cover 
- Documents
- Vector stores
- Retrievers


### Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has two attributes:

- page_content: a string representing the content;
- metadata: a dict containing arbitrary metadata.
The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual Document object often represents a chunk of a larger document.

Let's generate some sample documents:

In [1]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [2]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [19]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key=os.environ.get("GROQ_API_KEY")

os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

llm=ChatGroq(
    groq_api_key=groq_api_key,
    model="llama2-7b-chat",
)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, client=<groq.resources.chat.completions.Completions object at 0x000002069C4B4A10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002069C6EB450>, model_name='llama2-7b-chat', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [6]:
## Embedding using HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings.embed_documents([d.page_content for d in documents])

c:\Users\Rishabh Pandey\Desktop\GenAI\LANGCHAIN\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2088.04it/s]


[[-0.01731993816792965,
  -0.0054204100742936134,
  0.07044940441846848,
  0.08165137469768524,
  -0.07681580632925034,
  0.06437184661626816,
  0.015746787190437317,
  -0.03470795601606369,
  -0.0018970728851854801,
  0.04379239305853844,
  0.07946214824914932,
  0.013665755279362202,
  0.05030686780810356,
  0.050671327859163284,
  -0.008141128346323967,
  0.012151161208748817,
  0.0046973214484751225,
  -0.05843111872673035,
  -0.03880036249756813,
  -0.04363381862640381,
  -0.1595572829246521,
  -0.0439545102417469,
  0.04428236931562424,
  0.002594777848571539,
  -0.08631843328475952,
  -0.010869855061173439,
  0.03484111279249191,
  -0.0889279693365097,
  0.00971560925245285,
  -0.014673779718577862,
  -0.07247954607009888,
  0.02434421330690384,
  0.02227185294032097,
  0.04608333855867386,
  -0.04038286954164505,
  0.04511640593409538,
  0.04822223260998726,
  0.008071529679000378,
  0.005383200477808714,
  0.033374425023794174,
  0.012306132353842258,
  0.03212824463844299,
  

In [7]:
## VectorStores for converting text inot embeddings

from langchain_chroma import Chroma

vectorstore=Chroma.from_documents(documents,embedding=embeddings)
vectorstore



In [8]:
## we can do similarity search


vectorstore.similarity_search("cat")

[Document(id='b2d1559d-8a7e-46ba-9e5e-0754da517b53', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='457a8cca-a3e7-4d6b-85db-a81a9d2b50ee', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='9c289b58-3be7-46d6-8ab7-62fdbe662629', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='8f63207c-b0f9-4087-a02d-43562e464a38', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [9]:
## Async Querying for VectorStore
"""It means that it will not wait for the result to come back and 
will return a future object which can be awaited later. This is 
useful when you want to do multiple queries in parallel and wait
for all of them to complete."""

await vectorstore.asimilarity_search("cat")

[Document(id='b2d1559d-8a7e-46ba-9e5e-0754da517b53', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='457a8cca-a3e7-4d6b-85db-a81a9d2b50ee', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='9c289b58-3be7-46d6-8ab7-62fdbe662629', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='8f63207c-b0f9-4087-a02d-43562e464a38', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [10]:
### Similarity score calculation

vectorstore.similarity_search_with_score("cat")

[(Document(id='b2d1559d-8a7e-46ba-9e5e-0754da517b53', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351056814193726),
 (Document(id='457a8cca-a3e7-4d6b-85db-a81a9d2b50ee', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740898847579956),
 (Document(id='9c289b58-3be7-46d6-8ab7-62fdbe662629', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
  1.5956902503967285),
 (Document(id='8f63207c-b0f9-4087-a02d-43562e464a38', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
  1.6657923460006714)]

### Retrievers
LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method:

In [13]:

from langchain_core.documents import Document   

from langchain_core.runnables import RunnableLambda  # lambda function to wrap the similarity search method

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat", "dog"])  # for getting the top 1 result
## cat ke just baad dog ko bhi pass kar diya hai, so it will return the top 1 result for both cat and dog

[[Document(id='b2d1559d-8a7e-46ba-9e5e-0754da517b53', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='457a8cca-a3e7-4d6b-85db-a81a9d2b50ee', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

Vectorstores implement an as_retriever method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following:

In [14]:
## Same way we can also use the retriever method of vectorstore to get the top k results for a query
## also we have passed mulpiple queries in the batch method to get the top k results for each query
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["cat", "dog"])

[[Document(id='b2d1559d-8a7e-46ba-9e5e-0754da517b53', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='457a8cca-a3e7-4d6b-85db-a81a9d2b50ee', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [26]:
## RAG
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([("human", message)])

# Use a currently available Groq model
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=groq_api_key
)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
)

response = rag_chain.invoke("tell me about dogs")
print(response.content)


Dogs are great companions, known for their loyalty and friendliness.
